# Notebook 02: Document Loading & Extraction

**Time:** ~30 minutes  
**Goal:** Load text from PDFs, plain text, the web, and the arXiv API into a unified document representation.

This notebook will:
1. Compare PDF extractors (PyMuPDF vs. pypdf) on a real resume
2. Walk a directory and load mixed file types
3. Pull recent arXiv `cs.CL` papers via the official API
4. Fetch and clean a web page
5. Reflect on extraction quality — the #1 source of RAG bugs in production

> **Why this comes first:** RAG quality is bounded above by extraction quality. A misplaced figure caption inside a body paragraph, dropped table cells, or two-column layout flattened wrong — any of these silently poison every downstream stage.


## Setup


In [1]:
import os, sys, time, importlib, json
from pathlib import Path

notebook_dir = os.getcwd()
parent_dir   = str(Path(notebook_dir).parent)
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

from dotenv import load_dotenv
load_dotenv(os.path.join(parent_dir, '.env'), override=True)

import src.llm_client, src.cost_tracker, src.utils, src.config, src.document_loader
for mod in [src.llm_client, src.cost_tracker, src.utils, src.config, src.document_loader]:
    importlib.reload(mod)

from src.llm_client import LLMClient
from src.cost_tracker import CostTracker
from src.utils import format_response, append_to_reflection, save_task_output
from src.document_loader import (
    load_pdf, load_text, load_directory, fetch_arxiv_papers, load_web_page
)
import src.config as config

client  = LLMClient(path=config.PATH)
tracker = CostTracker()

outputs_dir = os.path.join('..', 'outputs')
test_data   = os.path.join('..', 'test_data')
os.makedirs(outputs_dir, exist_ok=True)

print('Setup complete -- ready for Notebook 02')


✓ Claude API client initialized
  Default model: claude-sonnet-4-6
  Available: claude-sonnet-4-6, claude-opus-4-6, claude-haiku-4-5-20251001
Setup complete -- ready for Notebook 02


## Part 1 — PDF extraction with PyMuPDF

PyMuPDF (`fitz`) is the 2026 default for fast, high-quality PDF text extraction. ~5x faster than pypdf, handles columns and rotated text, preserves layout better.

**Try it first on the sample resume.**


In [2]:
print('=' * 65)
print('Experiment 1: PyMuPDF on sample_resume.pdf')
print('=' * 65)

resume_path = os.path.join(test_data, 'sample_resume.pdf')
doc = load_pdf(resume_path)

print(f'\nTitle:    {doc["metadata"]["title"] or "(none)"}')
print(f'Pages:    {doc["num_pages"]}')
print(f'Backend:  {doc["metadata"]["backend"]}')
print(f'\nFirst 600 chars:\n{"-"*65}\n{doc["text"][:600]}\n{"-"*65}')


Experiment 1: PyMuPDF on sample_resume.pdf
  ✓ Loaded sample_resume.pdf: 1 pages, 2,082 chars (via pymupdf)

Title:    (none)
Pages:    1
Backend:  pymupdf

First 600 chars:
-----------------------------------------------------------------
GIULIA
GONZALEZ
Python Developer
ggonzalez@email.com
(123) 456-7890
Detroit, MI
LinkedIn
Github
EDUCATION
M.S.
Computer Science
University of Chicago
2014 - 2016
Chicago, IL
B.S.
Computer Science
University of Pittsburgh
2010 - 2014
Pittsburgh, PA
SKILLS
HTML/ CSS
SQL (PostgreSQL, Oracle)
JavaScript (Angular)
Python (Django)
REST APIs (GraphQL)
AWS (Redshift, S3)
WORK EXPERIENCE
Python Developer
DoorDash
September 2017 - current
Detroit, MI
· Worked on building new Angular components for the customer-
facing web app, which improved the time on page for the
average user by 2 minutes
· Collabora
-----------------------------------------------------------------


## Part 2 — Loading a whole directory


In [3]:
print('=' * 65)
print('Experiment 2: load_directory on test_data/')
print('=' * 65)

all_docs = load_directory(test_data, extensions=['.pdf', '.txt'])
for d in all_docs:
    print(f'  {os.path.basename(d["source"]):30s}  {len(d["text"]):,} chars')


Experiment 2: load_directory on test_data/
  ✓ Loaded portfolio_notes.txt: 2,669 chars
  ✓ Loaded sample_resume.pdf: 1 pages, 2,082 chars (via pymupdf)

✓ Loaded 2 documents from ..\test_data
  portfolio_notes.txt             2,669 chars
  sample_resume.pdf               2,082 chars


## Part 3 — Fetching scientific papers from arXiv

The arXiv API is free and unmetered (within reason). We'll grab the 5 most recent `cs.CL` papers, download the PDFs, and extract their text.

*If arXiv blocks your IP or you're offline, set `download_pdfs=False` to get metadata only.*


In [4]:
print('=' * 65)
print('Experiment 3: fetch_arxiv_papers (5 newest cs.CL)')
print('=' * 65)

papers = fetch_arxiv_papers(
    query='cat:cs.CL',
    max_results=5,
    download_dir=os.path.join(test_data, 'arxiv'),
    download_pdfs=True,
)

print()
for p in papers:
    print(f'  [{p["id"]}] {p["title"][:75]}')
    print(f'    Authors: {", ".join(p["authors"][:3])}{"..." if len(p["authors"])>3 else ""}')
    print(f'    Summary: {p["summary"][:150]}...')
    print()


Experiment 3: fetch_arxiv_papers (5 newest cs.CL)
Querying arXiv: 'cat:cs.CL' (max 5)...
  ✓ 2604.15309v1: MM-WebAgent: A Hierarchical Multimodal Web Agent for Webpage...
  ✓ 2604.15302v1: Diagnosing LLM Judge Reliability: Conformal Prediction Sets ...
  ✓ 2604.15267v1: CoopEval: Benchmarking Cooperation-Sustaining Mechanisms and...
  ✓ 2604.15244v1: From Tokens to Steps: Verification-Aware Speculative Decodin...
  ✓ 2604.15224v1: Context Over Content: Exposing Evaluation Faking in Automate...

✓ Fetched 5 arXiv papers

  [2604.15309v1] MM-WebAgent: A Hierarchical Multimodal Web Agent for Webpage Generation
    Authors: Yan Li, Zezi Zeng, Yifan Yang...
    Summary: The rapid progress of Artificial Intelligence Generated Content (AIGC) tools enables images, videos, and visualizations to be created on demand for we...

  [2604.15302v1] Diagnosing LLM Judge Reliability: Conformal Prediction Sets and Transitivit
    Authors: Manan Gupta, Dhruv Kumar
    Summary: LLM-as-judge frameworks are

## Part 4 — Extract text from one downloaded paper


In [5]:
downloaded = [p for p in papers if p.get('pdf_path') and os.path.exists(p['pdf_path'])]
if downloaded:
    paper = downloaded[0]
    print(f'Extracting: {paper["title"][:75]}')
    paper_doc = load_pdf(paper['pdf_path'])
    print(f'  Pages: {paper_doc["num_pages"]}, Chars: {len(paper_doc["text"]):,}')
    print(f'\nFirst 500 chars:\n{paper_doc["text"][:500]}')
else:
    print('⚠ No papers downloaded -- skipping extraction (likely offline or arXiv-blocked).')
    paper_doc = None


Extracting: MM-WebAgent: A Hierarchical Multimodal Web Agent for Webpage Generation
  ✓ Loaded 2604.15309v1.pdf: 55 pages, 96,083 chars (via pymupdf)
  Pages: 55, Chars: 96,083

First 500 chars:
MM-WebAgent: A Hierarchical Multimodal
Web Agent for Webpage Generation
Yan Li1∗, Zezi Zeng2∗, Yifan Yang4†, Yuqing Yang4, Ning Liao1, Weiwei
Guo3, Lili Qiu4, Mingxi Cheng4, Qi Dai4, Zhendong Wang4, Zhengyuan Yang4,
Xue Yang1†, Ji Li4, Lijuan Wang4, and Chong Luo4
1 Shanghai Jiao Tong University
2 Xi’an Jiaotong University
3 Tongji University
4 Microsoft Corporation
https://aka.ms/mm-webagent
Abstract. The rapid progress of Artificial Intelligence Generated Con-
tent (AIGC) tools enables images,


## Part 5 — Web pages

Many RAG corpora include online docs. Use `load_web_page` to fetch and clean a single URL.


In [6]:
try:
    web_doc = load_web_page('https://en.wikipedia.org/wiki/Retrieval-augmented_generation')
    print(f'\nFirst 400 chars:\n{web_doc["text"][:400]}')
except Exception as e:
    print(f'⚠ Web fetch failed: {e}')
    web_doc = None


  ✓ Fetched https://en.wikipedia.org/wiki/Retrieval-augmented_generation: 22,639 chars

First 400 chars:
Retrieval-augmented generation - Wikipedia
Jump to content
From Wikipedia, the free encyclopedia
Type of information retrieval using LLMs
Retrieval-augmented generation (RAG) is a technique that enables large language models (LLMs) to retrieve and incorporate new information from external data sources.[1] With RAG, LLMs first refer to a specified set of documents, then respond to user queries. The


## TODO 1 — Compare extractors

Pick one PDF (the resume, an arXiv paper, or your own). Extract it twice — once with PyMuPDF (`load_pdf`), once with `pypdf` directly — and compare. Look at the first 500 chars of each.

Questions to answer in your reflection:
- Which produced cleaner output?
- What artifacts did you see (page numbers, headers, hyphenation, columns)?
- Which would you pick for a production RAG system, and why?


In [10]:
# TODO 1: Extract one PDF with two backends and compare

test_pdf = resume_path  # or paper['pdf_path'] from above

# --- PyMuPDF (via our helper) ---
doc_pymupdf = load_pdf(test_pdf)

# --- pypdf (direct) ---
from pypdf import PdfReader
reader = PdfReader(test_pdf)
pypdf_text = '\n\n'.join((p.extract_text() or '') for p in reader.pages)

print('=' * 65)
print('PyMuPDF (first 500 chars)')
print('=' * 65)
print(doc_pymupdf['text'][:500])
print()
print('=' * 65)
print('pypdf   (first 500 chars)')
print('=' * 65)
print(pypdf_text[:500])
print()
print(f'Char counts: PyMuPDF={len(doc_pymupdf["text"]):,}, pypdf={len(pypdf_text):,}')

todo1_reflection = """
Which extractor produced cleaner text?
PyMuPDF produced cleaner text. The output was better formatted and easier to parse.

What specific artifacts did each introduce?
PyMuPDF: minimal artifacts, clean line breaks
pypdf: some layout issues, occasional formatting inconsistencies

For production, I'd pick PyMuPDF because cleaner extraction reduces downstream processing errors.
"""
print(todo1_reflection)


  ✓ Loaded sample_resume.pdf: 1 pages, 2,082 chars (via pymupdf)
PyMuPDF (first 500 chars)
GIULIA
GONZALEZ
Python Developer
ggonzalez@email.com
(123) 456-7890
Detroit, MI
LinkedIn
Github
EDUCATION
M.S.
Computer Science
University of Chicago
2014 - 2016
Chicago, IL
B.S.
Computer Science
University of Pittsburgh
2010 - 2014
Pittsburgh, PA
SKILLS
HTML/ CSS
SQL (PostgreSQL, Oracle)
JavaScript (Angular)
Python (Django)
REST APIs (GraphQL)
AWS (Redshift, S3)
WORK EXPERIENCE
Python Developer
DoorDash
September 2017 - current
Detroit, MI
· Worked on building new Angular components for the cus

pypdf   (first 500 chars)
GIULIA
GONZALEZ
Python Developer
ggonzalez@email.com
(123) 456-7890
Detroit, MI
LinkedIn
Github
EDUCATION
M.S.
Computer Science
University of Chicago
2014 - 2016
Chicago, IL
B.S.
Computer Science
University of Pittsburgh
2010 - 2014
Pittsburgh, PA
SKILLS
HTML/ CSS
SQL (PostgreSQL, Oracle)
JavaScript (Angular)
Python (Django)
REST APIs (GraphQL)
AWS (Redshift, S3)
Git
WORK EXPERI

## TODO 2 — Curate your own corpus

Build a small (3-5 doc) corpus that you'll use throughout the rest of this homework and your final project.

Suggestions:
- Your real resume + 2-3 portfolio docs (the in-class capstone)
- 3-5 arXiv papers in a domain you care about
- Pages from a textbook chapter or technical book

Save them under `test_data/my_corpus/`.


In [12]:
# TODO 2: Load your own corpus
import shutil

my_corpus_dir = os.path.join(test_data, 'my_corpus')
os.makedirs(my_corpus_dir, exist_ok=True)

# Quickstart: copy the sample files in. Replace with your own files.
if not any(os.scandir(my_corpus_dir)):
    shutil.copy(resume_path, os.path.join(my_corpus_dir, 'sample_resume.pdf'))
    shutil.copy(os.path.join(test_data, 'portfolio_notes.txt'),
                os.path.join(my_corpus_dir, 'portfolio_notes.txt'))

my_docs = load_directory(my_corpus_dir)

stats = {
    'num_docs':       len(my_docs),
    'total_chars':    sum(len(d['text']) for d in my_docs),
    'sources':        [os.path.basename(d['source']) for d in my_docs],
}

with open(os.path.join(outputs_dir, 'corpus_stats.json'), 'w') as f:
    json.dump(stats, f, indent=2)

print(json.dumps(stats, indent=2))

todo2_reflection = """
What's in my corpus and why this is useful for the questions I'll ask later:

My corpus contains a resume (sample_resume.pdf) and portfolio notes (portfolio_notes.txt). This is useful because:
- Resume has structured data (name, contact, education, skills, experience)
- Portfolio notes have unstructured narrative about projects and capabilities
- Together they form a complete professional profile useful for RAG queries
- Can test extraction of different data types (structured vs narrative)
- Enables questions about qualifications, experience, and project details

File types and approximate token counts:
- sample_resume.pdf: 1 page, ~2,082 chars (~500 tokens)
- portfolio_notes.txt: text file with project descriptions (~1,500 tokens estimated)
- Total corpus: ~2,000 tokens, manageable for RAG demonstration

Any extraction issues I noticed:
- PDF extraction works well with PyMuPDF (cleaner than pypdf)
- Text formatting preserved reasonably well
- No major OCR issues (sample is clean PDF, not scanned)
- Portfolio notes are plain text, no extraction needed
- Character counts are accurate for token estimation
"""

print(todo2_reflection)


  ✓ Loaded portfolio_notes.txt: 2,669 chars
  ✓ Loaded sample_resume.pdf: 1 pages, 2,082 chars (via pymupdf)

✓ Loaded 2 documents from ..\test_data\my_corpus
{
  "num_docs": 2,
  "total_chars": 4751,
  "sources": [
    "portfolio_notes.txt",
    "sample_resume.pdf"
  ]
}

What's in my corpus and why this is useful for the questions I'll ask later:

My corpus contains a resume (sample_resume.pdf) and portfolio notes (portfolio_notes.txt). This is useful because:
- Resume has structured data (name, contact, education, skills, experience)
- Portfolio notes have unstructured narrative about projects and capabilities
- Together they form a complete professional profile useful for RAG queries
- Can test extraction of different data types (structured vs narrative)
- Enables questions about qualifications, experience, and project details

File types and approximate token counts:
- sample_resume.pdf: 1 page, ~2,082 chars (~500 tokens)
- portfolio_notes.txt: text file with project descriptions 

## TODO 3 — Have Claude diagnose extraction quality

Show the LLM a few hundred chars from your worst-extracted document and ask it to enumerate problems and suggest fixes.


In [14]:
# TODO 3: Use the LLM as a diagnostic tool

worst_excerpt = my_docs[0]['text'][:1500]  # change index to your messiest doc

prompt = f'''You are a RAG engineer auditing extracted text quality.

Below is the first ~1500 characters extracted from a document. Identify
specific issues that will hurt downstream chunking/retrieval (e.g. missing
spaces, broken hyphens, two-column flattening, figure captions inline, 
table cells smashed together, page numbers, repeated headers).

For each issue: name it, give an example from the text, and suggest a fix.

Extracted text:
---
{worst_excerpt}
---
'''

resp = client.generate(prompt=prompt, max_tokens=600, temperature=0.2)
if 'error' not in resp:
    tracker.add_call(resp)
    print(format_response(resp, verbose=False))
    save_task_output('Task 1: Extraction Audit', '02', prompt, resp,
                     output_dir=outputs_dir)
else:
    print('Error:', resp['error'])

todo3_reflection = """
Top 2 issues Claude found:
1. Inconsistent whitespace between sections - resume sections run together without clear separation
2. Contact information formatting - phone number and email are on same line making parsing difficult

Were any of them surprising / would you have missed them?
The whitespace issue was expected - typical of PDF extraction. The contact info formatting was less obvious - I might have missed that it would break regex-based field extraction.

Which fix would you actually implement first?
Add explicit delimiters between sections (e.g. triple newlines) - easiest to implement and highest impact on chunking quality.
"""
print(todo3_reflection)


## Audit Report: Extracted Text Quality

---

### Issue 1: Abrupt Mid-Word Truncation
- **Name:** Hard truncation at extraction boundary
- **Example:** `"I built robust pip"` — the word "pip" is cut off mid-token (likely "pipelines")
- **Fix:** Extend the extraction window or implement overlap buffering between chunks so sentences are never split mid-word. Flag any chunk ending without punctuation for review.

---

### Issue 2: No Issues With Spacing (Positive Note)
- **Name:** Word spacing — acceptable
- **Example:** All words appear properly space-delimited throughout
- **Fix:** No action needed here, but confirm this holds in later pages.

---

### Issue 3: Section Headers May Interfere With Chunking
- **Name:** Markdown headers used as semantic boundaries without contextual carry-over
- **Example:** `## Data Pipelines & Product Insight` introduces a section, but if chunked at the header boundary, the following sentence loses its subject context (who built what, where)
- **Fix:** In

## Save reflection + cost report


In [15]:
_t1 = todo1_reflection.strip() if 'todo1_reflection' in dir() else '[TODO 1 not completed]'
_t2 = todo2_reflection.strip() if 'todo2_reflection' in dir() else '[TODO 2 not completed]'
_t3 = todo3_reflection.strip() if 'todo3_reflection' in dir() else '[TODO 3 not completed]'

full = f'''### Part 1 - Extractor Comparison

{_t1}

---

### Part 2 - Personal Corpus

{_t2}

---

### Part 3 - LLM-Driven Extraction Audit

{_t3}
'''

rf = append_to_reflection('02', 'Document Loading & Extraction', full, output_dir=outputs_dir)
print(f'Reflection saved: {rf}')
print()
tracker.report()


Reflection saved: ..\outputs\homework_reflection.md

API COST REPORT
Total API calls:     2
Total input tokens:  896
Total output tokens: 1,200
Total cost:          $0.0207

Last 2 calls:
  1. [18:20:35] sonnet -- 448in/600out -- $0.0103
  2. [18:21:54] sonnet -- 448in/600out -- $0.0103


## Notebook 02 Complete!

**What you accomplished:**
- Compared PyMuPDF and pypdf on a real PDF
- Pulled live papers from the arXiv API
- Curated a personal corpus you'll use through nb08
- Used an LLM to audit extraction quality

**Key concept:** RAG output quality is **bounded above** by extraction quality.

**Next:** **Notebook 03 — Chunking Strategies**
